In [2]:
import os
import numpy as np
import fitsio
from astropy.io import fits
from astropy.table import Table, vstack
from tqdm import tqdm
import math
import pandas as pd

# Format SGA 2020 Data

In [3]:
sga2020 = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', hdu=1)
df2020 = sga2020.to_pandas()
df2020 = df2020.rename(columns={"RA": "Target_RA", "DEC": "Target_DEC", "SGA_ID": "TargetID"})
df2020[:10].to_csv("/pscratch/sd/q/qshimp/SGA2020-data/sga2020_test.csv")
df2020

,TargetID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,...,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
0,2,b'SGA-2020 2',b'PGC1283207',1283207,228.377086,5.423202,b'S?',152.199997,0.363078,0.724436,...,0.264044,0.345595,3.303355,0.003811,15.195567,0.298264,0.300107,3.233377,0.011724,0
1,3,b'SGA-2020 3',b'PGC1310416',1310416,202.544438,6.934594,b'Sc',159.259995,0.401791,0.781628,...,0.876432,0.273606,2.499542,0.493439,15.235263,1.309869,0.178668,2.175050,0.203912,0
2,4,b'SGA-2020 4',b'SDSSJ145059.93+135143.0',4435547,222.749787,13.861911,b'S?',44.570000,0.333426,0.663743,...,0.488582,0.278250,3.214446,1.373326,16.807674,0.517704,0.322646,2.900518,1.805409,0
3,7,b'SGA-2020 7',b'PGC1742504',1742504,182.088808,25.602276,b'Sbc',84.970001,0.548277,0.251189,...,0.765731,0.522855,2.304599,0.006013,15.191324,1.040820,0.329563,2.152033,0.004725,0
4,18,b'SGA-2020 18',b'2MASXJ12340801+4535444',3550748,188.533553,45.595643,b'E',168.649994,0.530884,0.695024,...,0.324728,1.743924,2.177483,0.023486,15.124181,0.165473,1.469468,3.394879,0.028654,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383615,5005230,b'SGA-2020 5005230',b'DR8-0774m270-618',-1,77.397511,-27.106348,b'PSF',0.000000,0.332000,1.000000,...,0.231687,0.219425,3.111871,0.016901,16.022186,0.241948,0.201020,3.134553,0.011278,0
383616,5005238,b'SGA-2020 5005238',b'DR8-3541p242-2263',-1,354.025406,24.266276,b'PSF',0.000000,0.553000,1.000000,...,0.241669,0.313603,2.891678,0.068157,15.030018,0.215990,0.324286,2.999769,0.032022,0
383617,5005241,b'SGA-2020 5005241',b'DR8-3598p237-618',-1,359.778524,23.672069,b'PSF',0.000000,0.647000,1.000000,...,0.150522,0.867359,3.746000,0.005017,15.713827,0.251672,0.725019,3.094475,0.004562,0
383618,5005244,b'SGA-2020 5005244',b'DR8-1933p245-3598',-1,193.433580,24.573039,b'PSF',0.000000,0.508000,1.000000,...,8809.708984,0.000040,0.644355,0.112737,15.642013,9365.626953,0.000035,0.740361,0.139232,0


In [4]:
# Check for duplicate Target IDs
duplicates = df2020[df2020.duplicated(subset='TargetID', keep=False)]
print(f"{len(duplicates)} duplicate IDs")

0 duplicate IDs


In [5]:
def MergeNearCoords(df):
    # Round to the nearest thousandth of a degree (3.6")
    df['RA_round'] = df['Target_RA'].round(3)
    df['DEC_round'] = df['Target_DEC'].round(3)

    # Remove irrelevant coordinates
    duplicates = df[df.duplicated(subset=['RA_round', 'DEC_round'], keep="last")]
    uniqueObjects = df.drop(duplicates.index)
    return uniqueObjects
    
df2020 = MergeNearCoords(df2020)
df2020

,TargetID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,...,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT,RA_round,DEC_round
0,2,b'SGA-2020 2',b'PGC1283207',1283207,228.377086,5.423202,b'S?',152.199997,0.363078,0.724436,...,3.303355,0.003811,15.195567,0.298264,0.300107,3.233377,0.011724,0,228.377,5.423
1,3,b'SGA-2020 3',b'PGC1310416',1310416,202.544438,6.934594,b'Sc',159.259995,0.401791,0.781628,...,2.499542,0.493439,15.235263,1.309869,0.178668,2.175050,0.203912,0,202.544,6.935
2,4,b'SGA-2020 4',b'SDSSJ145059.93+135143.0',4435547,222.749787,13.861911,b'S?',44.570000,0.333426,0.663743,...,3.214446,1.373326,16.807674,0.517704,0.322646,2.900518,1.805409,0,222.750,13.862
3,7,b'SGA-2020 7',b'PGC1742504',1742504,182.088808,25.602276,b'Sbc',84.970001,0.548277,0.251189,...,2.304599,0.006013,15.191324,1.040820,0.329563,2.152033,0.004725,0,182.089,25.602
4,18,b'SGA-2020 18',b'2MASXJ12340801+4535444',3550748,188.533553,45.595643,b'E',168.649994,0.530884,0.695024,...,2.177483,0.023486,15.124181,0.165473,1.469468,3.394879,0.028654,0,188.534,45.596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383615,5005230,b'SGA-2020 5005230',b'DR8-0774m270-618',-1,77.397511,-27.106348,b'PSF',0.000000,0.332000,1.000000,...,3.111871,0.016901,16.022186,0.241948,0.201020,3.134553,0.011278,0,77.398,-27.106
383616,5005238,b'SGA-2020 5005238',b'DR8-3541p242-2263',-1,354.025406,24.266276,b'PSF',0.000000,0.553000,1.000000,...,2.891678,0.068157,15.030018,0.215990,0.324286,2.999769,0.032022,0,354.025,24.266
383617,5005241,b'SGA-2020 5005241',b'DR8-3598p237-618',-1,359.778524,23.672069,b'PSF',0.000000,0.647000,1.000000,...,3.746000,0.005017,15.713827,0.251672,0.725019,3.094475,0.004562,0,359.779,23.672
383618,5005244,b'SGA-2020 5005244',b'DR8-1933p245-3598',-1,193.433580,24.573039,b'PSF',0.000000,0.508000,1.000000,...,0.644355,0.112737,15.642013,9365.626953,0.000035,0.740361,0.139232,0,193.434,24.573


# Format SGA 2025 Data

In [6]:
sgaBeta2025 = Table.read('/global/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/SGA2025-beta-parent-refcat-v1.6.kd.fits', hdu = 1)
df2025 = sgaBeta2025.to_pandas()
df2025.rename(columns={"ra": "Target_RA", "dec": "Target_DEC", "ref_id": "TargetID"}, inplace=True)
df2025

,Target_RA,Target_DEC,TargetID,mag,fitmode,pa,ba,diam
0,219.925802,-32.530045,4176048,17.780001,0,70.355278,0.583230,0.555508
1,219.891999,-32.342994,4176016,17.709999,0,150.428970,0.460713,0.507900
2,219.940904,-32.331298,242036,17.840000,0,134.882629,0.220731,0.682237
3,220.209215,-32.010931,243654,18.040001,0,91.798492,0.703881,0.507167
4,220.362323,-31.994911,4257822,15.400000,0,10.682078,0.616908,1.066838
...,...,...,...,...,...,...,...,...
470620,30.146677,28.225196,5005050,12.831638,0,162.803253,0.342439,3.757807
470621,30.867045,28.274770,4957379,16.096416,0,84.813797,0.607179,0.527874
470622,30.360603,28.734268,4807432,14.230906,0,54.906796,0.786671,0.765473
470623,30.320490,28.836853,5054143,11.612162,0,1.327161,0.183276,10.501398


In [7]:
# Check for duplicate Ref IDs
duplicates = df2025[df2025.duplicated(subset='TargetID', keep=False)]
print(f"{len(duplicates)} duplicate IDs")

0 duplicate IDs


In [8]:
df2025 = MergeNearCoords(df2025)
df2025
# All objects are unique

,Target_RA,Target_DEC,TargetID,mag,fitmode,pa,ba,diam,RA_round,DEC_round
0,219.925802,-32.530045,4176048,17.780001,0,70.355278,0.583230,0.555508,219.926,-32.530
1,219.891999,-32.342994,4176016,17.709999,0,150.428970,0.460713,0.507900,219.892,-32.343
2,219.940904,-32.331298,242036,17.840000,0,134.882629,0.220731,0.682237,219.941,-32.331
3,220.209215,-32.010931,243654,18.040001,0,91.798492,0.703881,0.507167,220.209,-32.011
4,220.362323,-31.994911,4257822,15.400000,0,10.682078,0.616908,1.066838,220.362,-31.995
...,...,...,...,...,...,...,...,...,...,...
470620,30.146677,28.225196,5005050,12.831638,0,162.803253,0.342439,3.757807,30.147,28.225
470621,30.867045,28.274770,4957379,16.096416,0,84.813797,0.607179,0.527874,30.867,28.275
470622,30.360603,28.734268,4807432,14.230906,0,54.906796,0.786671,0.765473,30.361,28.734
470623,30.320490,28.836853,5054143,11.612162,0,1.327161,0.183276,10.501398,30.320,28.837


# Identify 2020 galxies within SGA 2025

In [9]:
# Implementing a Cone Search
from astropy.coordinates import SkyCoord
import astropy.units as u

# Create SkyCoord objects
coords2025 = SkyCoord(ra=sgaBeta2025["ra"] * u.deg,
                     dec=sgaBeta2025["dec"] * u.deg)

coords2020 = SkyCoord(ra=sga2020["RA"] * u.deg,
                      dec=sga2020["DEC"] * u.deg)

# Identify matches within SGA 2020
idx, d2d, d3d = coords2020.match_to_catalog_sky(coords2020, nthneighbor=2)

MAX_SEP = 10 * u.arcsec  # Maximum allowed separation on-sky for a duplicate
MAX_DIAM_DIFF = 0.5       # Tolerance for diameter (26 isophote) difference

matches = pd.DataFrame({
    'Source_Idx': np.arange(len(sga2020)),
    'Matched_To_Idx': idx,
    'Separation_arcsec': d2d.to(u.arcsec).value
})

duplicateFilter = (matches['Separation_arcsec'] <= MAX_SEP.value)              
valid_duplicates = matches[duplicateFilter]
print(f"Number of matched targets: {len(valid_duplicates)}")
# Do a cone search (cross-match) within 10 arcseconds
idx_2025, idx_2020, sep2d, _ = coords2025.search_around_sky(coords2020, 3.6 * u.arcsec)

# This gives you the indices of matches from both catalogs
print(f"Number of matched targets: {len(idx_2025)}")

Number of matched targets: 1668
Number of matched targets: 282326


In [10]:
# Function that adds a column that identifies data from one SGA set (giveID) in another set (receiveID)
def CompareDataset(df_receiveID, df_giveID):
    coords = ["RA_round", "DEC_round"]
    
    give_subset = df_giveID[coords + ["TargetID"]].copy()
    give_subset = give_subset.rename(columns={"TargetID": "SGA ID"})
    
    merged_df = pd.merge(df_receiveID, give_subset, on=coords, how='left')
    merged_df["SGA ID"] = merged_df["SGA ID"].astype("Int64")
    return merged_df
    
df2025Compared = CompareDataset(df2025, df2020)
df2025Compared

,Target_RA,Target_DEC,TargetID,mag,fitmode,pa,ba,diam,RA_round,DEC_round,SGA ID
0,219.925802,-32.530045,4176048,17.780001,0,70.355278,0.583230,0.555508,219.926,-32.530,<NA>
1,219.891999,-32.342994,4176016,17.709999,0,150.428970,0.460713,0.507900,219.892,-32.343,<NA>
2,219.940904,-32.331298,242036,17.840000,0,134.882629,0.220731,0.682237,219.941,-32.331,<NA>
3,220.209215,-32.010931,243654,18.040001,0,91.798492,0.703881,0.507167,220.209,-32.011,<NA>
4,220.362323,-31.994911,4257822,15.400000,0,10.682078,0.616908,1.066838,220.362,-31.995,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
470620,30.146677,28.225196,5005050,12.831638,0,162.803253,0.342439,3.757807,30.147,28.225,1250853
470621,30.867045,28.274770,4957379,16.096416,0,84.813797,0.607179,0.527874,30.867,28.275,1070616
470622,30.360603,28.734268,4807432,14.230906,0,54.906796,0.786671,0.765473,30.361,28.734,502445
470623,30.320490,28.836853,5054143,11.612162,0,1.327161,0.183276,10.501398,30.320,28.837,<NA>


In [11]:
df2025Cleaned = df2025Compared[df2025Compared['SGA ID'].isna()]
df2025Cleaned = df2025Cleaned.drop(columns=["SGA ID", "RA_round", "DEC_round"])
df2025Cleaned

,Target_RA,Target_DEC,TargetID,mag,fitmode,pa,ba,diam
0,219.925802,-32.530045,4176048,17.780001,0,70.355278,0.583230,0.555508
1,219.891999,-32.342994,4176016,17.709999,0,150.428970,0.460713,0.507900
2,219.940904,-32.331298,242036,17.840000,0,134.882629,0.220731,0.682237
3,220.209215,-32.010931,243654,18.040001,0,91.798492,0.703881,0.507167
4,220.362323,-31.994911,4257822,15.400000,0,10.682078,0.616908,1.066838
...,...,...,...,...,...,...,...,...
470599,29.748286,27.061345,3345171,13.347000,0,59.265774,0.714058,0.530304
470603,31.308315,27.139710,3888284,18.170000,0,85.974800,0.698627,0.416352
470608,31.454218,27.408196,3168451,16.190001,0,108.342827,0.860133,0.892261
470611,30.775969,27.506619,1333435,19.167000,0,100.055969,0.822534,0.456332


In [12]:
PIX_SCALE       = 0.262       # arcsec/pixel for Legacy Surveys
CUTOUT_SIZE_PIX = 152         # fixed 152x152 cutouts
scale = 1 # Manually adjust this to change allowed galaxy size
diamMax = scale*PIX_SCALE*CUTOUT_SIZE_PIX/60 # max diameter in arcmin
print(f"Cutout length/height: {diamMax} arcmin")

sizedTargets = df2025Cleaned[df2025Cleaned['diam'] <= diamMax]
print(f"{len(sizedTargets)} targets in sample")
bigs = df2025Cleaned[df2025Cleaned['diam'] > diamMax]
print(f"{len(bigs)} oversized galaxies removed")

Cutout length/height: 0.6637333333333333 arcmin
95930 targets in sample
98354 oversized galaxies removed


##### Interesting note
I mistakenly used 39.824 arcmin instead of arsec; this pulled 8 largest objects (still sorted by ID. 
First 3 have no galaxy, fourth and fifth are Magellanic Clouds, sixth is Sculptor Galaxy, 
seventh is a faint, large cloud of stars, eighth is Triangulum Galaxy

In [13]:
# Create table of 2025 SGA data
sizedTargets[:1000].to_csv("/pscratch/sd/q/qshimp/SGA2025-data/sga2025_sizedTargets1000.csv")
sizedTargets[:10]

,Target_RA,Target_DEC,TargetID,mag,fitmode,pa,ba,diam
0,219.925802,-32.530045,4176048,17.780001,0,70.355278,0.583230,0.555508
1,219.891999,-32.342994,4176016,17.709999,0,150.428970,0.460713,0.507900
3,220.209215,-32.010931,243654,18.040001,0,91.798492,0.703881,0.507167
5,219.972199,-31.718623,4180680,17.139999,0,92.536026,0.544621,0.561025
6,219.529606,-31.677126,245155,18.219999,0,60.774433,0.209309,0.588386
10,219.940608,-31.527629,4168989,12.671000,0,5.618719,0.985002,0.647525
15,220.196759,-31.466778,4181392,13.370000,0,160.237244,0.521736,0.615121
16,220.200082,-31.465109,4172482,12.966000,0,107.568214,0.671095,0.480799
18,220.453808,-31.415309,246182,17.049999,0,45.811050,0.484390,0.618283
21,219.902048,-31.279685,4177607,17.510000,0,78.755539,0.539636,0.581592


##### Copy this code to run fp_cutouts in galaxy morphology directory 
To activate desi

`source /global/common/software/desi/desi_environment.sh main`


`python make_cutouts.py --csv /pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_1000_sga152x152_irregular.csv --outdir /pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/Irregular`

Replace --csv file path with file path generated up above, replace --outdir file path with folder to store cutouts

# Append Anchor Points

In [14]:
# Format target set for h5
sga2025_152 = Table.read('/pscratch/sd/q/qshimp/Sorter/152by152_sga2025_1000test.fits')
#sga2025_152
dfTargets = Table()
dfTargets["targetid"] = sga2025_152["ref_id"]
dfTargets["target_ra"] = sga2025_152["ra"]
dfTargets["target_dec"] = sga2025_152["dec"]
dfTargets["g_mag"] = [np.nan]
dfTargets["z_mag"] = [np.nan]
dfTargets["r_mag"] = sga2025_152["r_mag"]
dfTargets["Morphology"] = [""]
dfTargets["T_type"] = [int()]
dfTargets["Main_type"] = 30
dfTargets["Path"] = sga2025_152["Path"]
dfTargets = dfTargets.to_pandas()
dfTargets

,targetid,target_ra,target_dec,g_mag,z_mag,r_mag,Morphology,T_type,Main_type,Path
0,4176048,219.925802,-32.530045,NaN,NaN,17.780,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/4...
1,4176016,219.891999,-32.342994,NaN,NaN,17.710,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/4...
2,243654,220.209215,-32.010931,NaN,NaN,18.040,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/2...
3,4180680,219.972199,-31.718623,NaN,NaN,17.140,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/4...
4,245155,219.529606,-31.677126,NaN,NaN,18.220,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/2...
...,...,...,...,...,...,...,...,...,...,...
995,242991,207.367422,-32.155268,NaN,NaN,17.940,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/2...
996,4167949,207.203656,-32.114381,NaN,NaN,17.040,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/4...
997,246238,207.097748,-31.404243,NaN,NaN,18.380,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/2...
998,3167090,208.658513,-30.900445,NaN,NaN,16.460,,0,30,b'/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/3...


In [15]:
# Format anchors for h5
anchors = Table.read("/pscratch/sd/q/qshimp/Sorter/152by152_sga2020_anchors.fits")
dfAnchors = anchors.to_pandas()
dfAnchors = dfAnchors.rename(columns={"ra": "target_ra", "dec": "target_dec", "ref_id": "targetid"})
#dfAnchors[:4000].to_csv("/pscratch/sd/q/qshimp/SGA2020-data/Anchors/VI_1000_sga152x152_irregular.csv")
dfAnchors

,targetid,target_ra,target_dec,g_mag,z_mag,r_mag,Morphology,T_type,Main_type,Path
0,831151,213.505901,31.148203,17.112806,15.652968,16.261892,b'E',-5.0,20,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...
1,841964,182.226242,60.719100,16.788475,15.383980,15.916330,b'E',-5.0,20,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...
2,742784,144.671469,51.826068,15.109658,13.753175,14.197537,b'E',-5.0,20,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...
3,1007845,183.183555,31.450164,16.120312,14.581675,15.226092,b'E',-5.0,20,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...
4,396832,200.344998,4.316768,17.439743,15.729994,16.446762,b'E',-5.0,20,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...
...,...,...,...,...,...,...,...,...,...,...
3995,920263,179.990971,49.564702,15.605430,15.147808,15.052292,b'I',10.0,-5,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...
3996,1064408,185.303178,8.963251,16.784065,15.790439,16.144346,b'I',10.0,-5,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...
3997,224150,186.525137,31.859704,16.642347,15.925266,16.206484,b'I',10.0,-5,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...
3998,1321251,163.462168,56.758492,17.104418,16.753347,16.765520,b'I',10.0,-5,b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...


In [28]:
withAnchors = pd.concat([dfTargets, dfAnchors], ignore_index=True)
withAnchors.loc[750, 'targetid'] = 2019760 # changing a conflicting pair of ref_id and sga_id
#print(withAnchors.loc[withAnchors['targetid'] == 201976]) # sga_id remains true
withAnchors['Path'] = withAnchors['Path'].str.decode('utf-8')
withAnchors.to_csv("/pscratch/sd/q/qshimp/Sorter/sga2025_1000test_with_anchors.csv")
withAnchors

      targetid   target_ra  target_dec      g_mag      z_mag      r_mag  \
2707    201976  246.578693   49.926554  16.851194  15.575987  16.065723   

     Morphology  T_type  Main_type  \
2707      b'Sb'     3.0         10   

                                                   Path  
2707  b'/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchor...  


,targetid,target_ra,target_dec,g_mag,z_mag,r_mag,Morphology,T_type,Main_type,Path
0,4176048,219.925802,-32.530045,NaN,NaN,17.780000,,0.0,30,/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/417...
1,4176016,219.891999,-32.342994,NaN,NaN,17.710000,,0.0,30,/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/417...
2,243654,220.209215,-32.010931,NaN,NaN,18.040000,,0.0,30,/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/243...
3,4180680,219.972199,-31.718623,NaN,NaN,17.140000,,0.0,30,/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/418...
4,245155,219.529606,-31.677126,NaN,NaN,18.220000,,0.0,30,/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/245...
...,...,...,...,...,...,...,...,...,...,...
4995,920263,179.990971,49.564702,15.605430,15.147808,15.052292,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...
4996,1064408,185.303178,8.963251,16.784065,15.790439,16.144346,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...
4997,224150,186.525137,31.859704,16.642347,15.925266,16.206484,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...
4998,1321251,163.462168,56.758492,17.104418,16.753347,16.765520,b'I',10.0,-5,/pscratch/sd/q/qshimp/Cutouts/sga2020/Anchors/...


In [61]:
path = withAnchors['Path'][0]
print(path)

/pscratch/sd/q/qshimp/Cutouts/sga2025/dr10/4176048_grz_152_219.9258_-32.5300.fits
